# 01 — Data Understanding

## Morocco Retail Intelligence

**Objective:** load the six raw datasets, understand their structure, inspect data quality, verify key relationships, and prepare reusable cleaning operations.

This notebook is for exploration and documentation. Reusable functions live in `src/morocco_retail_intelligence/data/`. Raw files must remain unchanged; cleaned outputs will later be written to `data/processed/`.

## 1. Imports and project paths

We use `Path` for portable paths and Pandas for tabular data. The path assumes the notebook runs from the `notebooks/` directory.

In [1]:
from pathlib import Path

import pandas as pd

from morocco_retail_intelligence.data.cleaning import (
    build_missing_values_report,
    fill_missing_values,
    remove_duplicate_rows,
    remove_duplicates_with_report,
)
from morocco_retail_intelligence.data.loader import load_data
from morocco_retail_intelligence.data.profiling import (
    build_dataset_overview,
    profile_dataframe,
)
from morocco_retail_intelligence.data.transformations import (
    convert_date_columns,
    convert_numeric_columns,
    normalize_text_columns,
)
from morocco_retail_intelligence.data.validation import (
    validate_foreign_key,
    validate_primary_key,
)


from morocco_retail_intelligence.data.pipeline import (
    save_cleaned_datasets,
)



## 2. Load the raw datasets

The loader returns a `RetailData` object containing six DataFrames.

In [2]:
RAW_DATA_DIR = Path("../data/raw")
data = load_data(RAW_DATA_DIR)

datasets = {
    "customers": data.customers,
    "products": data.products,
    "orders": data.orders,
    "order_items": data.order_items,
    "payments": data.payments,
    "reviews": data.reviews,
}

print(f"Loaded {len(datasets)} datasets.")

Loaded 6 datasets.


## 3. First look at the data

Use `head()`, `tail()`, and `sample()` to inspect records. `shape` gives row/column counts; `columns` lists fields; `dtypes` and `info()` show data types and non-null counts; `describe(include="all")` summarizes columns.

In [3]:
data.customers.head()

,customer_id,customer_age,city,customer_segment,signup_date
0,C0001,57.0,Meknes,Regular,2023-05-15
1,C0002,69.0,Fes,Regular,2025-05-07
2,C0003,54.0,Marrakech,Regular,2024-10-01
3,C0004,40.0,Casablanca,Regular,2024-04-22
4,C0005,35.0,Casablanca,Regular,2024-01-03


In [4]:
data.orders.sample(5, random_state=42)

,order_id,customer_id,order_date,order_status,delivery_date,shipping_city,shipping_fee_mad
610,O00611,C0888,2025-07-18,Delivered,2025-07-19,El Jadida,35.0
3048,O03049,C0147,2025-12-17,Delivered,2025-12-24,Marrakech,30.0
2480,O02481,C0566,2025-11-04,Delivered,2025-11-10,Meknes,30.0
1913,O01914,C0091,2024-01-23,Delivered,2024-01-24,Rabat,30.0
2883,O02884,C0069,2024-06-13,Delivered,2024-06-17,Fes,20.0


In [5]:
for name, df in datasets.items():
    print("\n" + "="*5 +" " + name.upper()  +" " + "=" * 5)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    # display(df.head(3))


===== CUSTOMERS =====
Shape: (1000, 5)
Columns: ['customer_id', 'customer_age', 'city', 'customer_segment', 'signup_date']

===== PRODUCTS =====
Shape: (100, 6)
Columns: ['product_id', 'product_name', 'category', 'unit_price_mad', 'cost_price_mad', 'stock_status']

===== ORDERS =====
Shape: (3205, 7)
Columns: ['order_id', 'customer_id', 'order_date', 'order_status', 'delivery_date', 'shipping_city', 'shipping_fee_mad']

===== ORDER_ITEMS =====
Shape: (5376, 7)
Columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price_mad', 'discount_rate', 'line_total_mad']

===== PAYMENTS =====
Shape: (3200, 5)
Columns: ['payment_id', 'order_id', 'payment_method', 'payment_status', 'payment_amount_mad']

===== REVIEWS =====
Shape: (2019, 6)
Columns: ['review_id', 'order_id', 'customer_id', 'rating', 'review_date', 'review_comment']


## 4. Dataset-level overview

`build_dataset_overview()` summarizes row counts, column counts, exact duplicate rows, and memory usage.

In [6]:
dataset_overview = build_dataset_overview(datasets)
dataset_overview

,dataset,rows,columns,duplicate_rows,memory_mb
0,customers,1000,5,0,0.22
1,products,100,6,0,0.02
2,orders,3205,7,5,1.05
3,order_items,5376,7,0,1.01
4,payments,3200,5,0,0.71
5,reviews,2019,6,0,0.57


## 5. Column-level profiling

The profile reports data types, non-null counts, missing counts/percentages, and distinct-value counts. `nunique()` excludes missing values by default.

In [7]:
orders_profile = profile_dataframe(data.orders)
orders_profile

,data_type,non_null_count,missing_count,missing_percentage,unique_values
order_id,str,3205,0,0.00,3200
customer_id,str,3205,0,0.00,966
order_date,str,3205,0,0.00,720
order_status,str,3205,0,0.00,4
delivery_date,str,2632,573,17.88,717
shipping_city,str,3205,0,0.00,12
shipping_fee_mad,float64,3187,18,0.56,7


In [8]:
profiles = {name: profile_dataframe(df) for name, df in datasets.items()}
for name, profile in profiles.items():
    print(f"\n{name.upper()}")
    display(profile)


CUSTOMERS


,data_type,non_null_count,missing_count,missing_percentage,unique_values
customer_id,str,1000,0,0.0,1000
customer_age,float64,988,12,1.2,53
city,str,1000,0,0.0,12
customer_segment,str,1000,0,0.0,3
signup_date,str,1000,0,0.0,635



PRODUCTS


,data_type,non_null_count,missing_count,missing_percentage,unique_values
product_id,str,100,0,0.0,100
product_name,str,100,0,0.0,100
category,str,100,0,0.0,5
unit_price_mad,float64,100,0,0.0,100
cost_price_mad,float64,100,0,0.0,100
stock_status,str,100,0,0.0,3



ORDERS


,data_type,non_null_count,missing_count,missing_percentage,unique_values
order_id,str,3205,0,0.00,3200
customer_id,str,3205,0,0.00,966
order_date,str,3205,0,0.00,720
order_status,str,3205,0,0.00,4
delivery_date,str,2632,573,17.88,717
shipping_city,str,3205,0,0.00,12
shipping_fee_mad,float64,3187,18,0.56,7


... (2 outputs left out)


PAYMENTS


,data_type,non_null_count,missing_count,missing_percentage,unique_values
payment_id,str,3200,0,0.0,3200
order_id,str,3200,0,0.0,3200
payment_method,str,3200,0,0.0,4
payment_status,str,3200,0,0.0,3
payment_amount_mad,float64,3200,0,0.0,3025



REVIEWS


,data_type,non_null_count,missing_count,missing_percentage,unique_values
review_id,str,2019,0,0.00,2019
order_id,str,2019,0,0.00,2019
customer_id,str,2019,0,0.00,868
rating,int64,2019,0,0.00,5
review_date,str,2019,0,0.00,702
review_comment,str,2004,15,0.74,6


## 6. Inspect categorical values

`value_counts(dropna=False)` includes missing values and helps reveal inconsistent spelling, capitalization, whitespace, or unexpected categories. Use exact column names from `df.columns`.

In [9]:
for column in ["status", "city"]:
    if column in data.orders.columns:
        print(f"\nOrders — {column}")
        display(data.orders[column].value_counts(dropna=False))

In [10]:
for dataset_name, column in [("customers", "segment"), ("payments", "payment_method"), ("payments", "payment_status")]:
    df = datasets[dataset_name]
    if column in df.columns:
        print(f"\n{dataset_name} — {column}")
        display(df[column].value_counts(dropna=False))


payments — payment_method


payment_method
Cash on Delivery    1343
Card                1221
Bank Transfer        367
Wallet               269
Name: count, dtype: int64


payments — payment_status


payment_status
Paid        2965
Refunded     167
Failed        68
Name: count, dtype: int64

## 7. Understand dataset relationships

A primary key should be non-null and unique. A foreign key should reference an existing key in its parent dataset. Update any field names here if your CSV uses different names.

In [11]:
primary_keys = {
    "customers": (data.customers, "customer_id"),
    "products": (data.products, "product_id"),
    "orders": (data.orders, "order_id"),
    "order_items": (data.order_items, "order_item_id"),
    "payments": (data.payments, "payment_id"),
    "reviews": (data.reviews, "review_id"),
}

primary_key_report = pd.DataFrame([
    validate_primary_key(df, column, name)
    for name, (df, column) in primary_keys.items()
])
primary_key_report

,dataset,column,total_rows,missing_keys,duplicate_keys,is_valid
0,customers,customer_id,1000,0,0,True
1,products,product_id,100,0,0,True
2,orders,order_id,3205,0,5,False
3,order_items,order_item_id,5376,0,0,True
4,payments,payment_id,3200,0,0,True
5,reviews,review_id,2019,0,0,True


If `order_items` uses a different unique identifier, replace `order_item_id` with its actual column name.

In [12]:
foreign_key_relationships = [
    (data.orders, "customer_id", data.customers, "customer_id", "orders → customers"),
    (data.order_items, "order_id", data.orders, "order_id", "order_items → orders"),
    (data.order_items, "product_id", data.products, "product_id", "order_items → products"),
    (data.payments, "order_id", data.orders, "order_id", "payments → orders"),
    (data.reviews, "order_id", data.orders, "order_id", "reviews → orders"),
    (data.reviews, "customer_id", data.customers, "customer_id", "reviews → customers"),
]

foreign_key_report = pd.DataFrame([
    validate_foreign_key(child, child_col, parent, parent_col, relationship)
    for child, child_col, parent, parent_col, relationship in foreign_key_relationships
])
foreign_key_report

,relationship,missing_foreign_keys,invalid_foreign_keys,is_valid
0,orders → customers,0,0,True
1,order_items → orders,0,0,True
2,order_items → products,0,0,True
3,payments → orders,0,0,True
4,reviews → orders,0,0,True
5,reviews → customers,0,0,True


### Why key validation matters

A join can silently lose records or multiply rows if keys are invalid or the relationship is misunderstood. `orders` to `order_items` is one-to-many. Joining order-level amounts to item-level rows can duplicate those amounts and inflate totals.

## 8. Inspect and convert dates

CSV date columns may be loaded as strings. `convert_date_columns()` converts selected columns to Pandas datetime. With `errors="coerce"`, invalid values become `NaT` and should be audited.

In [13]:
customers_dates = convert_date_columns(data.customers, ["signup_date"])
orders_dates = convert_date_columns(data.orders, ["order_date", "delivery_date"])
reviews_dates = convert_date_columns(data.reviews, ["review_date"])

display(orders_dates[["order_date", "delivery_date"]].dtypes)
display(orders_dates[["order_date", "delivery_date"]].head())

order_date       datetime64[us]
delivery_date    datetime64[us]
dtype: object

,order_date,delivery_date
0,2024-04-09,2024-04-13
1,2025-11-19,2025-11-23
2,2024-12-13,2024-12-20
3,2024-08-21,2024-08-24
4,2024-11-22,2024-11-23


In [14]:
orders_dates["order_year"] = orders_dates["order_date"].dt.year
orders_dates["order_month"] = orders_dates["order_date"].dt.month
orders_dates["order_quarter"] = orders_dates["order_date"].dt.quarter
orders_dates["order_weekday"] = orders_dates["order_date"].dt.day_name()
orders_dates[["order_date", "order_year", "order_month", "order_quarter", "order_weekday"]].head()

,order_date,order_year,order_month,order_quarter,order_weekday
0,2024-04-09,2024,4,2,Tuesday
1,2025-11-19,2025,11,4,Wednesday
2,2024-12-13,2024,12,4,Friday
3,2024-08-21,2024,8,3,Wednesday
4,2024-11-22,2024,11,4,Friday


Other useful datetime operations: `.dt.year`, `.dt.month`, `.dt.day`, `.dt.quarter`, `.dt.day_name()`, `.dt.strftime("%Y-%m")`. Subtract datetime columns to calculate durations.

## 9. Duplicate-row analysis

An exact duplicate repeats every column value. This differs from duplicate identifiers: two records can share an ID while containing different data. Remove exact duplicates from a copy and keep a report.

In [15]:
duplicate_summary = pd.DataFrame([
    {"dataset": name, "duplicate_rows": int(df.duplicated().sum())}
    for name, df in datasets.items()
])
duplicate_summary

,dataset,duplicate_rows
0,customers,0
1,products,0
2,orders,5
3,order_items,0
4,payments,0
5,reviews,0


In [16]:
orders_without_exact_duplicates, orders_duplicate_report = remove_duplicates_with_report(data.orders)
orders_duplicate_report

{'original_rows': 3205, 'cleaned_rows': 3200, 'removed_rows': 5}

`drop_duplicates()` options: `keep="first"` (default), `keep="last"`, `keep=False` (remove every row in duplicate groups), and `subset=[...]` (compare selected columns). Never overwrite raw data.

## 10. Missing-value analysis

Missing values require business-aware decisions. Do not automatically fill every missing value with zero or an average.

In [17]:
missing_reports = [
    build_missing_values_report(df, name)
    for name, df in datasets.items()
]
missing_values_report = pd.concat(missing_reports, ignore_index=True)
missing_values_report

,dataset,column,missing_count,missing_percentage
0,customers,customer_age,12,1.20
1,customers,customer_id,0,0.00
2,customers,city,0,0.00
3,customers,customer_segment,0,0.00
4,customers,signup_date,0,0.00
5,products,product_id,0,0.00
6,products,product_name,0,0.00
7,products,category,0,0.00
8,products,unit_price_mad,0,0.00
9,products,cost_price_mad,0,0.00


### Initial missing-value policy

- Customer age: preserve missing unless a defensible business rule exists.
- Order shipping fee: investigate because it affects financial analysis; do not assume zero.
- Review comment: missing comment does not mean missing rating. Preserve it, or use a display label only in presentation.

`fill_missing_values(df, fill_values)` returns a copy and fills only explicitly supplied columns. Example policy (do not run unless chosen):

In [18]:
reviews_display = fill_missing_values(data.reviews, {"comment": "No comment"})

## 11. Numeric type conversion

`pd.to_numeric()` converts numeric-looking values. `errors="coerce"` turns invalid values into missing values. Pandas nullable `Int64` supports integer values plus missing values. Verify column names against the CSVs before running this cell.

In [19]:
customers_numeric = convert_numeric_columns(data.customers, columns=["customer_age"], integer_columns=["customer_age"])
products_numeric = convert_numeric_columns(
    data.products,
    columns=["unit_price_mad", "cost_price_mad"]
    
)
orders_numeric = convert_numeric_columns(data.orders, columns=["shipping_fee_mad"])
order_items_numeric = convert_numeric_columns(
    data.order_items,
    columns=[ "unit_price_mad", "discount_rate", "line_total_mad","quantity"],
    integer_columns=["quantity"],
)
payments_numeric = convert_numeric_columns(data.payments, columns=["payment_amount_mad"])

display(products_numeric.dtypes)
display(order_items_numeric.dtypes)

product_id            str
product_name          str
category              str
unit_price_mad    float64
cost_price_mad    float64
stock_status          str
dtype: object

order_item_id         str
order_id              str
product_id            str
quantity            Int64
unit_price_mad    float64
discount_rate     float64
line_total_mad    float64
dtype: object

Only include columns that exist and are intended to be numeric. Integer conversion is appropriate only for whole-number values. Coercion can hide invalid source values by turning them into missing values; the production pipeline should record conversion failures before replacing original values.

In [20]:
data.orders.columns

Index(['order_id', 'customer_id', 'order_date', 'order_status',
       'delivery_date', 'shipping_city', 'shipping_fee_mad'],
      dtype='str')

## 12. Text normalization

Normalize accidental whitespace and optionally standardize case. This is useful for city names, segments, statuses, and payment methods. Do not lowercase names or comments when their capitalization should be preserved.

In [21]:
customers_text = normalize_text_columns(
    data.customers, columns=["city"],
    lowercase_columns=["city"],
)
orders_text = normalize_text_columns(
    data.orders, columns=["shipping_city", "order_status"],
    lowercase_columns=["shipping_city", "order_status"],
)
payments_text = normalize_text_columns(
    data.payments, columns=["payment_method", "payment_status"],
    lowercase_columns=["payment_method", "payment_status"],
)

display(orders_text["order_status"].value_counts(dropna=False))
display(orders_text["shipping_city"].value_counts(dropna=False))

order_status
delivered     2632
cancelled      256
returned       167
processing     150
Name: count, dtype: Int64

shipping_city
casablanca    747
marrakech     466
rabat         338
tangier       313
fes           274
agadir        232
kenitra       190
meknes        176
oujda         162
tetouan       151
el jadida      95
safi           61
Name: count, dtype: Int64

Reusable string methods: `.str.strip()` removes surrounding whitespace; `.str.lower()` / `.str.upper()` standardize case; `.str.replace()` replaces text/patterns; `.str.contains()` searches text; `.str.startswith()` checks prefixes; `.unique()` and `.value_counts(dropna=False)` inspect distinct values and frequencies.

## 13. Data-understanding conclusions

After running the notebook, record the actual findings below. Do not write conclusions until you have inspected the outputs.

- Which datasets contain exact duplicate rows?
- Which columns contain missing values, and how significant are they?
- Are expected primary keys valid?
- Are there invalid or missing foreign-key references?
- Are date columns parsed correctly?
- Did numeric conversion introduce new missing values?
- Are category or city labels inconsistent?

**Next step:** turn the reviewed decisions into a repeatable, auditable cleaning pipeline that writes to `data/processed/` while preserving `data/raw/`.

In [22]:
from morocco_retail_intelligence.data.cleaning_config import (
    DATASET_CONFIGS,
)

for dataset_name, config in DATASET_CONFIGS.items():
    print(f"\n{dataset_name.upper()}")
    print("Dates:", config.date_columns)
    print("Numbers:", config.numeric_columns)
    print("Text:", config.text_columns)
    print("Remove duplicates:", config.remove_exact_duplicates)


CUSTOMERS
Dates: ['signup_date']
Numbers: ['customer_age']
Text: ['city', 'customer_segment']
Remove duplicates: False

PRODUCTS
Dates: []
Numbers: ['unit_price_mad', 'cost_price_mad']
Text: ['category']
Remove duplicates: False

ORDERS
Dates: ['order_date', 'delivery_date']
Numbers: ['shipping_fee_mad']
Text: ['shipping_city', 'order_status']
Remove duplicates: True

ORDER_ITEMS
Dates: []
Numbers: ['quantity', 'unit_price_mad', 'discount_rate', 'line_total_mad']
Text: []
Remove duplicates: False

PAYMENTS
Dates: []
Numbers: ['payment_amount_mad']
Text: ['payment_method', 'payment_status']
Remove duplicates: False

REVIEWS
Dates: ['review_date']
Numbers: ['rating']
Text: []
Remove duplicates: False


In [23]:
from morocco_retail_intelligence.data.cleaning_config import (
    DATASET_CONFIGS,
)
from morocco_retail_intelligence.data.pipeline import clean_dataframe

orders_clean, orders_report = clean_dataframe(
    df=data.orders,
    dataset_name="orders",
    config=DATASET_CONFIGS["orders"],
)

orders_report

{'dataset': 'orders',
 'original_rows': 3205,
 'cleaned_rows': 3200,
 'duplicates_removed': 5,
 'missing_values_before': 591,
 'missing_values_after': 591,
 'date_conversion_failures': {'order_date': 0, 'delivery_date': 0},
 'numeric_conversion_failures': {'shipping_fee_mad': 0}}

In [24]:
data.reviews.columns

Index(['review_id', 'order_id', 'customer_id', 'rating', 'review_date',
       'review_comment'],
      dtype='str')

In [25]:
print("Original shape:", data.orders.shape)
print("Cleaned shape:", orders_clean.shape)


Original shape: (3205, 7)
Cleaned shape: (3200, 7)


In [26]:
cleaned_datasets = {}
cleaning_reports = []

for dataset_name, df in datasets.items():

    cleaned_df, report = clean_dataframe(
        df=df,
        dataset_name=dataset_name,
        config=DATASET_CONFIGS[dataset_name],
    )

    cleaned_datasets[dataset_name] = cleaned_df
    cleaning_reports.append(report)

cleaning_summary = pd.DataFrame(cleaning_reports)

cleaning_summary

,dataset,original_rows,cleaned_rows,duplicates_removed,missing_values_before,missing_values_after,date_conversion_failures,numeric_conversion_failures
0,customers,1000,1000,0,12,12,{'signup_date': 0},{'customer_age': 0}
1,products,100,100,0,0,0,{},"{'unit_price_mad': 0, 'cost_price_mad': 0}"
2,orders,3205,3200,5,591,591,"{'order_date': 0, 'delivery_date': 0}",{'shipping_fee_mad': 0}
3,order_items,5376,5376,0,0,0,{},"{'quantity': 0, 'unit_price_mad': 0, 'discount..."
4,payments,3200,3200,0,0,0,{},{'payment_amount_mad': 0}
5,reviews,2019,2019,0,15,15,{'review_date': 0},{'rating': 0}


In [27]:
from morocco_retail_intelligence.data.cleaning import (
    build_missing_values_report,
)

processed_missing_reports = [
    build_missing_values_report(df, name)
    for name, df in cleaned_datasets.items()
]

processed_missing_summary = pd.concat(
    processed_missing_reports,
    ignore_index=True,
)

processed_missing_summary[
    processed_missing_summary["missing_count"] > 0
]

,dataset,column,missing_count,missing_percentage
0,customers,customer_age,12,1.20
11,orders,delivery_date,573,17.91
12,orders,shipping_fee_mad,18,0.56
30,reviews,review_comment,15,0.74


In [28]:

PROCESSED_DATA_DIR = Path("../data/processed")

save_cleaned_datasets(
    cleaned_datasets=cleaned_datasets,
    cleaning_reports=cleaning_reports,
    output_dir=PROCESSED_DATA_DIR,
)

Saved: ..\data\processed\customers.csv
Saved: ..\data\processed\products.csv
Saved: ..\data\processed\orders.csv
Saved: ..\data\processed\order_items.csv
Saved: ..\data\processed\payments.csv
Saved: ..\data\processed\reviews.csv
Saved audit report: ..\data\processed\cleaning_report.csv


In [29]:
from morocco_retail_intelligence.data.validation import (
    validate_primary_key,
    validate_foreign_key,
)

primary_keys = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "payments": "payment_id",
    "reviews": "review_id",
}

primary_key_report = pd.DataFrame([
    validate_primary_key(
        df=cleaned_datasets[dataset_name],
        column=column,
        dataset_name=dataset_name,
    )
    for dataset_name, column in primary_keys.items()
])

primary_key_report

,dataset,column,total_rows,missing_keys,duplicate_keys,is_valid
0,customers,customer_id,1000,0,0,True
1,products,product_id,100,0,0,True
2,orders,order_id,3200,0,0,True
3,order_items,order_item_id,5376,0,0,True
4,payments,payment_id,3200,0,0,True
5,reviews,review_id,2019,0,0,True


In [30]:
foreign_key_relationships = [
    (
        "orders → customers",
        "orders", "customer_id",
        "customers", "customer_id",
    ),
    (
        "order_items → orders",
        "order_items", "order_id",
        "orders", "order_id",
    ),
    (
        "order_items → products",
        "order_items", "product_id",
        "products", "product_id",
    ),
    (
        "payments → orders",
        "payments", "order_id",
        "orders", "order_id",
    ),
    (
        "reviews → orders",
        "reviews", "order_id",
        "orders", "order_id",
    ),
    (
        "reviews → customers",
        "reviews", "customer_id",
        "customers", "customer_id",
    ),
]

foreign_key_report = pd.DataFrame([
    validate_foreign_key(
        child_df=cleaned_datasets[child_name],
        child_column=child_column,
        parent_df=cleaned_datasets[parent_name],
        parent_column=parent_column,
        relationship_name=relationship,
    )
    for (
        relationship,
        child_name, child_column,
        parent_name, parent_column,
    ) in foreign_key_relationships
])

foreign_key_report

,relationship,missing_foreign_keys,invalid_foreign_keys,is_valid
0,orders → customers,0,0,True
1,order_items → orders,0,0,True
2,order_items → products,0,0,True
3,payments → orders,0,0,True
4,reviews → orders,0,0,True
5,reviews → customers,0,0,True


In [31]:
validation_summary = pd.DataFrame([
    {
        "validation": "Primary keys",
        "total_checks": len(primary_key_report),
        "passed": int(primary_key_report["is_valid"].sum()),
        "failed": int((~primary_key_report["is_valid"]).sum()),
    },
    {
        "validation": "Foreign keys",
        "total_checks": len(foreign_key_report),
        "passed": int(foreign_key_report["is_valid"].sum()),
        "failed": int((~foreign_key_report["is_valid"]).sum()),
    },
])

validation_summary

,validation,total_checks,passed,failed
0,Primary keys,6,6,0
1,Foreign keys,6,6,0


In [32]:
invalid_primary_keys = primary_key_report[
    ~primary_key_report["is_valid"]
]

invalid_foreign_keys = foreign_key_report[
    ~foreign_key_report["is_valid"]
]

display(invalid_primary_keys)
display(invalid_foreign_keys)

,dataset,column,total_rows,missing_keys,duplicate_keys,is_valid


,relationship,missing_foreign_keys,invalid_foreign_keys,is_valid


In [33]:
export_validation = []

for dataset_name, original_df in cleaned_datasets.items():

    exported_df = pd.read_csv(
        PROCESSED_DATA_DIR / f"{dataset_name}.csv"
    )

    export_validation.append({
        "dataset": dataset_name,
        "expected_rows": len(original_df),
        "exported_rows": len(exported_df),
        "same_columns": (
            original_df.columns.tolist()
            == exported_df.columns.tolist()
        ),
        "rows_match": len(original_df) == len(exported_df),
    })

export_validation = pd.DataFrame(export_validation)

export_validation

,dataset,expected_rows,exported_rows,same_columns,rows_match
0,customers,1000,1000,True,True
1,products,100,100,True,True
2,orders,3200,3200,True,True
3,order_items,5376,5376,True,True
4,payments,3200,3200,True,True
5,reviews,2019,2019,True,True
